In [ ]:
df.head()

In [ ]:
percentile_table = pd.read_csv("2024_pitch_metrics_ranks.csv")
percentile_table.head()

In [ ]:
# ... existing code ...

# Add these pandas display options at the beginning of your notebook, after the imports
pd.set_option('display.max_rows', None)  # Show all rows
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', None)  # Width of the display in characters
pd.set_option('display.max_colwidth', None)  # Show full content of each column

# ... existing code ...
# df_transformed.head(250)

In [ ]:
file_path = r"C:\Users\TrevorWhite\Downloads\CSU Bakersfield - Pitch Metrics.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(f"Error: File '{file_path}' not found.")

dfd = pd.read_csv(file_path)

dfd.head()

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.backends.backend_pdf as pdf_backend
import pandas as pd
import numpy as np
import json
from sklearn.ensemble import RandomForestRegressor
from matplotlib.patches import Ellipse
import matplotlib.colors as mcolors
from joblib import load

# -------------------------------
# Define Functions
# -------------------------------

def load_csv_data(file_path):
    """
    Load the CSV data and validate the file exists.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Error: File '{file_path}' not found.")
    return pd.read_csv(file_path)

def coerce_numeric_columns(df, columns):
    """
    Coerce specified columns to numeric, in place.
    """
    df[columns] = df[columns].apply(pd.to_numeric, errors="coerce")
    return df

def check_required_columns(df, required_columns):
    """
    Check if the required columns are present in the DataFrame.
    """
    missing_cols = [col for col in required_columns if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns: {missing_cols}")

def fix_pitch_types(df):
    """
    Replace 'FS' with 'FA' in pitchType column.
    """
    df["pitchType"] = df["pitchType"].replace("FS", "FA")
    return df

def scale_release_coordinates(df):
    """
    Scale RelX and RelZ by dividing by 12.
    """
    df["RelX"] = df["RelX"] / 12
    df["RelZ"] = df["RelZ"] / 12
    return df

def load_and_apply_model(df, joblib_path):
    """
    Load the trained model from a joblib file, then predict shoulder_x and shoulder_z.
    Handles flipping for left-handed pitchers as well.
    """
    if not os.path.exists(joblib_path):
        raise FileNotFoundError(f"Error: Model file '{joblib_path}' not found.")

    model = load(joblib_path)
    mask_left = df["pitcherHand"] == "L"
    df.loc[mask_left, "RelX"] = df.loc[mask_left, "RelX"] * -1

    features = ["RelX", "RelZ"]
    df[features] = df[features].apply(pd.to_numeric, errors="coerce")

    df[["shoulder_x", "shoulder_z"]] = model.predict(df[features])
    df.loc[mask_left, ["RelX", "shoulder_x"]] *= -1

    return df

def define_pitch_colors():
    """
    Define custom colors for pitch types.
    """
    return {
        "FA": "#2ca02c",  # Green
        "FS": "#2ca02c",  # Green (Same as FA)
        "SI": "#FFD700",  # Gold
        "FC": "#FF8C00",  # Orange
        "SL": "#FF6347",  # Light Red
        "CH": "#1f77b4",  # Blue
        "CU": "#8B0000",  # Dark Red
        "KN": "#FF69B4"
    }

def define_plot_settings():
    """
    Define the base plot settings for the multi-page layout.
    """
    pitchers_per_page = 3
    page_figsize = (16, 12)
    row_ratios = [0.65, 0.35] * pitchers_per_page
    col_ratios_row1 = [0.8, 0.8, 1, 1, 1, 1]
    col_ratios_row2 = [0.8, 0.8, 1, 1, 1, 1]
    return pitchers_per_page, page_figsize, row_ratios, col_ratios_row1, col_ratios_row2

def plot_data_by_pitcher(df, pitchers_per_page, page_figsize, row_ratios, col_ratios_row1, col_ratios_row2, pitch_colors):
    """
    Create and show plots per group of pitchers. 
    """
    pitchers = df["pitcherAbbrevName"].unique()

    for page_start in range(0, len(pitchers), pitchers_per_page):
        fig = plt.figure(figsize=page_figsize)
        outer = gridspec.GridSpec(
            nrows=pitchers_per_page * 2,
            ncols=1,
            height_ratios=row_ratios,
            hspace=0.5
        )

        for i, pitcher in enumerate(pitchers[page_start:page_start + pitchers_per_page]):
            pitcher_data = df[df["pitcherAbbrevName"] == pitcher]
            pitcher_data = pitcher_data[
                np.abs(pitcher_data["RelX"] - pitcher_data["RelX"].median()) <= 3 * pitcher_data["RelX"].std()
            ]
            pitcher_data = pitcher_data[
                np.abs(pitcher_data["RelZ"] - pitcher_data["RelZ"].median()) <= 3 * pitcher_data["RelZ"].std()
            ]

            row_top = i * 2
            row_bottom = row_top + 1

            usage_pct = pitcher_data["pitchType"].value_counts(normalize=True) * 100
            count_00_pct = (
                pitcher_data[pitcher_data["count"] == '0-0']["pitchType"].value_counts(normalize=True) * 100
            )
            risp_pct = (
                pitcher_data[
                    (pitcher_data["ManOn2nd"] == 1) | (pitcher_data["ManOn3rd"] == 1)
                ]["pitchType"].value_counts(normalize=True) * 100
            )

            # ---------------------------
            # Row 1: Primary Visuals (65% height)
            # ---------------------------
            gs_row1 = gridspec.GridSpecFromSubplotSpec(
                nrows=1,
                ncols=6,
                subplot_spec=outer[row_top],
                width_ratios=col_ratios_row1,
                wspace=0.4
            )

            # Col 1: Scatterplot of RelX vs RelZ
            ax1 = fig.add_subplot(gs_row1[0])
            colors = pitcher_data["pitchType"].map(pitch_colors).fillna("gray")
            sc = ax1.scatter(
                pitcher_data["RelX"],
                pitcher_data["RelZ"],
                c=colors,
                alpha=0.7,
                edgecolors="black"
            )

            pitcher_hand = (
                pitcher_data["pitcherHand"].iloc[0]
                if "pitcherHand" in pitcher_data.columns
                else "Unknown"
            )
            handedness = "LHP" if pitcher_hand == "L" else "RHP" if pitcher_hand == "R" else "N/A"

            avg_shoulder_x = pitcher_data["shoulder_x"].mean()
            avg_shoulder_z = pitcher_data["shoulder_z"].mean()
            avg_rel_x = pitcher_data["RelX"].mean()
            avg_rel_z = pitcher_data["RelZ"].mean()

            ax1.scatter(
                avg_shoulder_x,
                avg_shoulder_z,
                color="black",
                s=100,
                alpha=0.3,
                label="Avg Shoulder Loc"
            )

            ax1.plot(
                [avg_shoulder_x, avg_rel_x],
                [avg_shoulder_z, avg_rel_z],
                linestyle="-",
                linewidth=5,
                color="gray",
                alpha=0.8
            )

            ax1.set_title(f"{pitcher} ({handedness})")
            ax1.set_xlabel("RelSide")
            ax1.set_ylabel("RelHeight")
            ax1.set_xlim(-3, 3)
            ax1.set_ylim(2, 7.5)

            # Col 2: Transposed Table for Pitch Usage %
            ax2 = fig.add_subplot(gs_row1[1])
            ax2.axis("off")

            table_data = []
            for pitch in usage_pct.index:
                table_data.append([
                    pitch,
                    f"{usage_pct.get(pitch, 0):.0f}%",
                    f"{count_00_pct.get(pitch, 0):.0f}%",
                    f"{risp_pct.get(pitch, 0):.0f}%"
                ])

            col_labels = ["Pitch", "%", "0-0", "RISP"]
            table = ax2.table(
                cellText=table_data,
                colLabels=col_labels,
                cellLoc="left",
                loc="center",
                bbox=[-0.2, 0, 1.1, 1]
            )
            table.auto_set_font_size(False)
            table.set_fontsize(10)
            table.scale(1.2, 1.2)

            # Confidence Ellipse Plots for Top 4 Pitch Types (Cols 3–6)
            pitch_usage = pitcher_data.groupby("pitchType").size().sort_values(ascending=False)
            top4 = pitch_usage.head(4).index.tolist()

            for idx, col in enumerate(range(2, 6)):
                ax = fig.add_subplot(gs_row1[col])
                if idx < len(top4):
                    pitch_type = top4[idx]
                    pitch_subset = pitcher_data[pitcher_data["pitchType"] == pitch_type]

                    horz = pitch_subset["HorzApprAngle"].dropna()
                    vert = pitch_subset["VertApprAngle"].dropna()

                    pitch_subset.loc[:, "Vel"] = pd.to_numeric(pitch_subset["Vel"], errors="coerce")
                    velo_10th = int(pitch_subset["Vel"].quantile(0.25)) if not pitch_subset["Vel"].isna().all() else 0
                    velo_95th = int(pitch_subset["Vel"].quantile(0.75)) if not pitch_subset["Vel"].isna().all() else 0

                    if len(horz) < 2 or len(vert) < 2:
                        ax.scatter(
                            horz,
                            vert,
                            color=pitch_colors.get(pitch_type, "gray"),
                            alpha=0.7,
                            edgecolors="black"
                        )
                        ax.set_title(f"{pitch_type}: {velo_10th}-{velo_95th}")
                        ax.set_xlabel("HorzApprAngle")
                        ax.set_ylabel("VertApprAngle")
                        ax.set_xlim(7, -7)
                        ax.set_ylim(-12, -2)
                        ax.text(
                            5.5, -2.3, "RHB <-", fontsize=8,
                            ha="center", va="center",
                            bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.5")
                        )
                        ax.text(
                            -5.5, -2.3, "-> LHB", fontsize=8,
                            ha="center", va="center",
                            bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.5")
                        )
                        continue

                    mean = [horz.mean(), vert.mean()]
                    cov = np.cov(horz, vert)
                    vals, vecs = np.linalg.eigh(cov)
                    order = vals.argsort()[::-1]
                    vals, vecs = vals[order], vecs[:, order]
                    theta = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
                    width, height = 2 * np.sqrt(vals)
                    color = pitch_colors.get(pitch_type, "gray")

                    ellipse = Ellipse(
                        xy=mean,
                        width=width,
                        height=height,
                        angle=theta,
                        edgecolor=color,
                        facecolor=color,
                        lw=2.5,
                        alpha=0.3
                    )
                    ax.add_patch(ellipse)

                    # percentile_table reference (as in original code)
                    ref_point = percentile_table[
                        (percentile_table["PitchType"] == pitch_type) &
                        (percentile_table["pitcherHand"] == pitcher_hand)
                    ][["HAA", "VAA"]]

                    if not ref_point.empty:
                        ref_haa = ref_point["HAA"].mean()
                        ref_vaa = ref_point["VAA"].mean()
                        base_color = mcolors.to_rgb(pitch_colors.get(pitch_type, "gray"))
                        darkened_color = tuple(c * 0.6 for c in base_color)
                        ax.plot(ref_haa, ref_vaa, marker="o", color=darkened_color, markersize=8)

                    ax.set_title(f"{pitch_type}: {velo_10th}-{velo_95th}")
                    ax.set_xlabel("HAA", labelpad=-2)
                    ax.set_ylabel("VAA", labelpad=-2)
                    ax.set_xlim(7, -7)
                    ax.set_ylim(-12, -2)
                    ax.text(
                        5.5, -2.3, "RHB <-", fontsize=8,
                        ha="center", va="center",
                        bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.5")
                    )
                    ax.text(
                        -5.5, -2.3, "-> LHB", fontsize=8,
                        ha="center", va="center",
                        bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.5")
                    )
                    ax.text(
                        0, -2.3, "● = NCAA Avg", fontsize=6,
                        ha="center", va="center",
                        bbox=dict(facecolor="white", edgecolor="black", boxstyle="round,pad=0.5"),
                        color=color
                    )

                    if pitcher_hand == "L":
                        haa_label_x = -5.5
                    else:
                        haa_label_x = 5.5

                    ax.text(
                        haa_label_x, -5, "↑", fontsize=14,
                        ha="center", va="center", color="gray"
                    )
                    ax.text(
                        haa_label_x, -3.9, "Flat", fontsize=8,
                        ha="center", va="center", color="gray"
                    )
                    ax.text(
                        haa_label_x, -10.4, "↓", fontsize=14,
                        ha="center", va="center", color="gray"
                    )
                    ax.text(
                        haa_label_x, -11.4, "Steep", fontsize=8,
                        ha="center", va="center", color="gray"
                    )
                else:
                    ax.text(0.5, 0.5, "No Data", ha='center', va='center')
                    ax.set_xticks([])
                    ax.set_yticks([])

            # ---------------------------
            # Row 2: Notes & Text Summaries (35% height)
            # ---------------------------
            gs_row2 = gridspec.GridSpecFromSubplotSpec(
                nrows=1, 
                ncols=6, 
                subplot_spec=outer[row_bottom],
                width_ratios=col_ratios_row2,
                wspace=0.4
            )

            # Cols 1-2: Handwritten Notes
            ax_notes = fig.add_subplot(gs_row2[:, :2])
            ax_notes.set_title("Notes")
            ax_notes.text(0.5, 0.5, "Write notes here...", ha='center', va='center', fontsize=12, color='gray')
            ax_notes.set_xticks([])
            ax_notes.set_yticks([])
            for spine in ax_notes.spines.values():
                spine.set_visible(False)

            # Cols 3-6: Minimal HAA/VAA Text Implementation
            from textwrap import fill
            FASTBALLS = ["FA"]
            BREAKING_PITCHES = ["SL", "CU", "FC"]
            TAILING_OFFSPEED = ["CH", "SI", "KN"]

            def get_haa_direction(diff_haa_in, pitcher_hand):
                """
                For fastballs/breaking (if you want to keep sign-based logic):
                RHP, diff>0 => 'towards RHB'; RHP, diff<0 => 'away from RHB'; etc.
                
                But for tailing offspeed, we'll override below so it always says
                RHP => 'towards RHB', LHP => 'away from RHB'.
                """
                if pitcher_hand == "R":
                    return "towards RHB" if diff_haa_in > 0 else "away from RHB"
                else:
                    return "away from RHB" if diff_haa_in > 0 else "towards RHB"

            for idx, col in enumerate(range(2, 6)):
                ax_text = fig.add_subplot(gs_row2[col])
                if idx < len(top4):
                    pitch_type = top4[idx]
                    pitch_subset = pitcher_data[pitcher_data["pitchType"] == pitch_type]

                    avg_haa = pitch_subset["HorzApprAngle"].mean()
                    avg_vaa = pitch_subset["VertApprAngle"].mean()

                    ref_data = percentile_table[
                        (percentile_table["PitchType"] == pitch_type)
                        & (percentile_table["pitcherHand"] == pitcher_hand)
                    ]
                    
                    if not ref_data.empty:
                        ref_haa = ref_data["HAA"].mean()
                        ref_vaa = ref_data["VAA"].mean()

                        diff_haa_in = (avg_haa - ref_haa) / 12
                        diff_vaa_in = (avg_vaa - ref_vaa) / 12

                        is_fastball = pitch_type in FASTBALLS
                        is_tailing_offspeed = pitch_type in TAILING_OFFSPEED
                        is_breaking = pitch_type in BREAKING_PITCHES
                        is_lefty = (pitcher_hand == "L")

                        # 1) Horizontal direction
                        haa_dir = get_haa_direction(diff_haa_in, pitcher_hand)

                        # 2) Movement word & “more/less”
                        if is_fastball:
                            # Example: keep it simple
                            movement_word = "moves"
                            more_less = "more"  # always "more" for fastballs

                        elif is_tailing_offspeed:
                            # Hardcode direction based on handedness, ignoring sign for direction
                            # but FLIP the "more/less" logic for lefty vs righty:
                            movement_word = "tails"
                            if is_lefty:
                                haa_dir = "away from RHB"
                                # If diff>0 => "less", else => "more"
                                more_less = "less" if diff_haa_in > 0 else "more"
                            else:
                                haa_dir = "towards RHB"
                                # If diff>0 => "more", else => "less"
                                more_less = "more" if diff_haa_in > 0 else "less"

                        else:  # Breaking pitch
                            movement_word = "breaks"
                            # Example flip for lefty vs righty
                            if is_lefty:
                                if diff_haa_in > 0:
                                    more_less = "more"
                                else:
                                    more_less = "less"
                            else:
                                if diff_haa_in > 0:
                                    more_less = "less"
                                else:
                                    more_less = "more"

                        # 3) Vertical approach angle
                        vaa_dir = "flatter" if diff_vaa_in > 0 else "steeper"

                        # 4) Build text
                        text_haa = (
                            f"{pitch_type} {movement_word} {abs(diff_haa_in):.2f} in. "
                            f"{more_less} than NCAA avg. {haa_dir}."
                        )
                        text_vaa = (
                            f"{pitch_type} is {vaa_dir} than NCAA avg. "
                            f"by {abs(diff_vaa_in):.2f} in."
                        )

                        wrapped_text_haa = fill(text_haa, width=30)
                        wrapped_text_vaa = fill(text_vaa, width=30)

                        ax_text.text(0.5, 0.7, wrapped_text_haa, ha='center', va='center', wrap=True)
                        ax_text.text(0.5, 0.2, wrapped_text_vaa, ha='center', va='center', wrap=True)
                    else:
                        ax_text.text(0.5, 0.5, "No reference data", ha='center', va='center')
                else:
                    ax_text.text(0.5, 0.5, "No Data", ha='center', va='center')

                ax_text.set_xticks([])
                ax_text.set_yticks([])
                for spine in ax_text.spines.values():
                    spine.set_visible(False)










        plt.show()

# -------------------------------
# Main Execution Flow
# -------------------------------
if __name__ == "__main__":
    # 1. Fixed File Path and Data Loading
    file_path = r"C:\Users\TrevorWhite\Downloads\UC Irvine - Pitch Metrics.csv"
    df = load_csv_data(file_path)

    num_cols = ["RelX", "RelZ", "ManOn2nd", "ManOn3rd", "VertApprAngle", "HorzApprAngle"]
    df = coerce_numeric_columns(df, num_cols)

    required_columns = [
        "pitcherAbbrevName", "pitchType", "RelX", "RelZ", "count",
        "ManOn2nd", "ManOn3rd", "VertApprAngle", "HorzApprAngle"
    ]
    check_required_columns(df, required_columns)

    df = fix_pitch_types(df)

    # 2. Model Loading & Prediction
    df = scale_release_coordinates(df)
    joblib_path = r"C:\Users\TrevorWhite\Downloads\shoulder_loc.joblib"
    df = load_and_apply_model(df, joblib_path)

    # 3. Define Custom Colors
    PITCH_COLORS = define_pitch_colors()

    # 4. Plot Settings
    pitchers_per_page, page_figsize, row_ratios, col_ratios_row1, col_ratios_row2 = define_plot_settings()

    # NOTE: percentile_table must be defined in the environment. 
    # Example placeholder:
    # percentile_table = pd.DataFrame({
    #     "PitchType": [...],
    #     "pitcherHand": [...],
    #     "HAA": [...],
    #     "VAA": [...]
    # })

    plot_data_by_pitcher(
        df,
        pitchers_per_page,
        page_figsize,
        row_ratios,
        col_ratios_row1,
        col_ratios_row2,
        PITCH_COLORS
    )


In [ ]:
for pitch_type in ["FA", "SL", "FC", "CH"]:
    norm = get_dynamic_norm(pitch_type, "chase_pct")
    print(f"{pitch_type} - vmin: {norm.vmin}, vcenter: {norm.vcenter}, vmax: {norm.vmax}")


In [ ]:
for pitch_type in ["FA", "SL", "FC", "CH"]:
    norm = get_dynamic_norm(pitch_type, "chase_pct")
    print(f"{pitch_type} - vmin: {norm.vmin}, vcenter: {norm.vcenter}, vmax: {norm.vmax}")
